# Lưu CSA real preparation — đúng Output126 thành Dataset

**Accelerator None, Internet ON**, không cần Add Input. Save & Run All.
Bật Kaggle Secret KAGGLE_API_TOKEN; nếu secret tên codex, sửa SECRET_NAME ở cell đầu.
Không dán token vào code/chat. OWNER phải có quyền xem source126 và tạo Dataset.
Chỉ tạo Dataset **Private mới**, không ghi đè dataset đã tồn tại.

Tải đúng version126, kiểm SHA đầy đủ8file, bao gồm regions.npz; không lấy latest,
không chạy lại POS/ROI hay mô hình. Giữ nguyên word lists như được tạo:
mục rid18575 đang cần kiểm tra từ gốc trước bước CLIP text, chưa tự sửa/chuẩn y.
Giữ Output126 đến khi Dataset xử lý xong.


In [ ]:
from pathlib import Path
KERNEL_VERSION='ducanh2403/2-vi-cai-kia-qua-nhieu/126'
OWNER='ducanh2403'
SLUG='csa-paper-real-preparation-v126'
TITLE='CSA real preparation v126 dataset'
SECRET_NAME='KAGGLE_API_TOKEN'
DESTINATION=Path('/kaggle/working/csa_real_preparation_dataset')


In [ ]:
import os,re,subprocess,sys
from kaggle_secrets import UserSecretsClient
if KERNEL_VERSION!='ducanh2403/2-vi-cai-kia-qua-nhieu/126':raise ValueError('This notebook pins126; do not substitute latest or another version.')
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('TITLE/SLUG must have6–50characters')
try:
    token=UserSecretsClient().get_secret(SECRET_NAME).strip()
    if not token:raise ValueError('empty')
except Exception:raise RuntimeError('Enable the configured SECRET_NAME in Add-ons > Secrets. Do not paste token in code/chat.') from None
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(key,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    with subprocess.Popen(args,env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace') as process:
        for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
        code=process.wait()
    if code:raise RuntimeError(f'Command failed ({code}); see redacted output. No automatic fallback or dataset overwrite.')

run_cli([sys.executable,'-m','pip','install','--upgrade','--force-reinstall','--quiet','https://github.com/Kaggle/kaggle-cli/archive/6894864ff4095cf98e9e36d3a694ff5fc9ba3818.zip'],authenticated=False)
VERSION_PREFLIGHT="import tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\n\n# Exercise the installed download implementation without contacting Kaggle.\napi = KaggleApi()\nwith tempfile.TemporaryDirectory() as directory, patch.object(api, 'build_kaggle_client') as factory:\n    endpoint = factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n    endpoint.return_value = SimpleNamespace(files=[], next_page_token=None, log=None)\n    api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/126', path=directory)\n    request = endpoint.call_args.args[0]\n    assert getattr(request, 'version_label', None) == 'v126', 'Installed CLI ignores the requested version; STOP before download.'\n    assert request.user_name == 'ducanh2403'\n    assert request.kernel_slug == '2-vi-cai-kia-qua-nhieu'\nprint('Version routing preflight PASS: request.version_label=v126; no network call.', flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib,json
INVENTORY={'kernel_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/126', 'source_review_zip_sha256': '9bd5a8b1e643186fa001a20d2782705898227fac54590b2ca468b8adf53a18b8', 'files': {'vocabulary.json': '924be210aa0daa8b0968a6d69ad2c66b3de603f49c35f4aa67247c75d6173fc7', 'bank_words.json': '450fc969e0842ca3270d1f899b7e8fe594639260644ad5815a69430fe48264b1', 'pilot_metadata.json': '892e90561dc99b7ca0e4e58615a5a750d11e6df0ce84d4c1482a9322a5c00beb', 'regions.npz': '054f79bcf13de02255893d5741e695ebfb453f66bc5e488234fba65702904b6c', 'receipt.json': 'b95196eba5dc19ef534485a0855b3ea194097ade05a1d44095194a6548fa1e42', 'summary.json': '206cfc43053179d65f9696debfd6b549f8f59a556a79b28c1dd69e42db2a8207', 'source_receipt.json': '8cc88416da138a7f2248d6371e6c1f03f67d3329fbf1334bb34289921daceabb', 'software_freeze.txt': '117f207643e1c4e13fea4f596069e16c15dca6927f7eca793a2b8ad38fb09b45'}}
SOURCE_CODE='"""Byte-preserving CPU transport only; no API, NumPy archive or model loading."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\n\nNAMES = {\'vocabulary.json\', \'bank_words.json\', \'pilot_metadata.json\', \'regions.npz\',\n         \'receipt.json\', \'summary.json\', \'source_receipt.json\', \'software_freeze.txt\'}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 << 20), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef package(source, destination, inventory):\n    expected = inventory[\'files\']\n    if inventory[\'kernel_version\'] != \'ducanh2403/2-vi-cai-kia-qua-nhieu/126\' or set(expected) != NAMES:\n        raise ValueError(\'Transport requires exact version126/eight original files\')\n    if any(not re.fullmatch(\'[0-9a-f]{64}\', h) for h in expected.values()):\n        raise ValueError(\'Invalid pinned digest\')\n    source, destination = Path(source).resolve(), Path(destination).resolve()\n    if source == destination or source in destination.parents or destination in source.parents:\n        raise ValueError(\'Separate non-nested folders required\')\n    for name, digest in expected.items():\n        if (source/name).is_symlink() or not (source/name).is_file() or sha(source/name) != digest:\n            raise ValueError(\'Missing/changed source payload: \'+name)\n    receipt = json.loads((source/\'receipt.json\').read_bytes())\n    summary = json.loads((source/\'summary.json\').read_bytes())\n    if receipt[\'status\'] != \'CPU_PREPARATION_COMPLETE\' or summary[\'status\'] != \'CPU_PREPARATION_COMPLETE\':\n        raise ValueError(\'Source is not completed CPU preparation\')\n    if receipt[\'schema\'] != \'csa_paper_words_roi_v1\' or receipt[\'semantic_embeddings_created\'] or receipt[\'training_ready\']:\n        raise ValueError(\'Unexpected source schema or training state\')\n    if summary[\'regions\'][\'images\'] != 320 or summary[\'regions\'][\'train_images\'] != 256 or summary[\'regions\'][\'val_images\'] != 64:\n        raise ValueError(\'Wrong pilot cohort\')\n    if receipt[\'payload_sha256\'] != {n: expected[n] for n in (\'vocabulary.json\', \'bank_words.json\', \'pilot_metadata.json\', \'regions.npz\')}:\n        raise ValueError(\'Receipt manifest differs\')\n    destination.mkdir(parents=True, exist_ok=True)\n    if any(p.is_symlink() or p.name not in NAMES | {\'transport_receipt.json\'} for p in destination.iterdir()):\n        raise ValueError(\'Preserve unrelated/partial destination; choose a fresh folder\')\n    needed = sum((source/n).stat().st_size for n in expected if not (destination/n).exists())\n    if shutil.disk_usage(destination).free < needed + (128 << 20):\n        raise RuntimeError(\'Insufficient copy space\')\n    for name, digest in expected.items():\n        target = destination/name\n        if target.exists():\n            if not target.is_file() or sha(target) != digest:\n                raise ValueError(\'Changed destination preserved: \'+name)\n            continue\n        partial = destination/(name+\'.partial\')\n        shutil.copyfile(source/name, partial)\n        if sha(partial) != digest:\n            raise ValueError(\'Copy digest differs: \'+name)\n        partial.replace(target)\n    report = dict(status=\'DATASET_FOLDER_READY\', source_kernel_version=inventory[\'kernel_version\'],\n        source_review_zip_sha256=inventory[\'source_review_zip_sha256\'], files=expected,\n        full_original_file_hashes_verified=True, original_bytes_unchanged=True,\n        model_executed=False, semantic_embeddings_created=False, semantic_word_lists_cleared=False,\n        lexical_warning=\'rid18575 vs ride881: suspected lemma problem, CPU token trace required before CLIP text encoding\')\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report, indent=2)+\'\\n\', encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='600cb085ef388d46bcdaf089990759724076d52135e2453c66c3c20c04e896b7'
exec(compile(SOURCE_CODE,'package_real_dataset.py','exec'))


In [ ]:
import tempfile,shutil
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<512*1024**2:
    raise RuntimeError('Need512MiB free for staged download/copy; preserve existing files.')
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa-real-v126-',dir='/tmp'))
pattern=r'(^|/)(?:'+'|'.join(re.escape(n) for n in INVENTORY['files'])+r')$'
print('Downloading EXACT Output:',KERNEL_VERSION,flush=True)
run_cli([sys.executable,'-m','kaggle','kernels','output',KERNEL_VERSION,'-p',str(DOWNLOAD),'--file-pattern',pattern])
matches=[p.parent for p in DOWNLOAD.rglob('receipt.json') if sha(p)==INVENTORY['files']['receipt.json']]
if len(matches)!=1:raise RuntimeError(f'Output126 returned {len(matches)} matching receipts. STOP, no latest fallback.')
report=package(matches[0],DESTINATION,INVENTORY)
print('Eight unchanged files verified/copied. Lexical warning retained:',report['lexical_warning'],flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private research assets preserved from exact Output126: TRAIN noun/verb POS word lists and320-image pooled raw CLIP regions. Upstream model/data terms retained. Source word list linguistic review pending (rid vs ride); not CLIP text embeddings or a trained caption model. No image pixels included. Original receipt/provenance preserved.')
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; choose a fresh destination.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be pending. Do not repeatedly create.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
try:run_cli([sys.executable,'-m','kaggle','datasets','status',f'{OWNER}/{SLUG}'])
except RuntimeError:print('Status not confirmed; check Dataset page before retrying.',flush=True)


Gửi path thực tế sau khi attach Dataset. Mount dự kiến:
`/kaggle/input/datasets/ducanh2403/csa-paper-real-preparation-v126`.
Đây chỉ là lưu dữ liệu, không mã hóa CLIP text hoặc sửa bank từ.
Codex chỉ kiểm metadata/static/mock, chưa gọi API hoặc chạy notebook này.
